In [1]:
# imports and settings

import os
import time
import pickle
import warnings
import pandas as pd
import networkx as nx

import numpy as np
from numpy import linalg as LA
from numpy import histogram2d

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.metrics import auc

import utils as ut
%load_ext autoreload
%autoreload 2

# do not show warnings
warnings.filterwarnings("ignore")

print("Imports complete.")

Imports complete.


---

## Detectors vs SNR - Sanity Check

In [10]:
# simulation parameters
fs = 128000  # sampling frequency
f0 = 640  # signal frequency
duration = 60  # seconds
snr_values = [0, -2, -4, -6, -8, -10, -12]  # from 0 dB to -8 dB

# detector args
nperseg = 8000
nfft = None # 2*nperseg
noverlap = 0.25
window = 'hanning'
dc = 20
crop_freq = 2000
norm_size = 9
quantization_bins = 3
# welch_threshold = 0.034
# s2g_threshold_laplacian = 0.24
# s2g_threshold_wasserstein = 16

raw_signal = ut.simulate_raw_signal(f0, fs, duration)
F, T, raw_Sxx, raw_phasogram = ut.calc_spectrogram(raw_signal, fs, nfft=nfft, nperseg=nperseg, percent_overlap=noverlap/nperseg, window=window, remove_dc=dc, crop_freq=crop_freq)

welch_detector = ut.WelchDetector(fs=fs, nperseg=nperseg, overlap=noverlap, nfft=nfft, window=window, dc=dc, crop_freq=crop_freq, norm_size=norm_size)
talmon_detector = ut.TalmonDetector(fs=fs, nperseg=nperseg, overlap=noverlap, nfft=nfft, window=window, dc=dc, crop_freq=crop_freq, norm_size=norm_size)
s2g_detector_laplacian = ut.S2GDetector(fs=fs, nperseg=nperseg, overlap=noverlap, nfft=nfft, window=window, dc=dc, crop_freq=crop_freq, quantization_levels=quantization_bins, mode="laplacian")
s2g_detector_wasserstein = ut.S2GDetector(fs=fs, nperseg=nperseg, overlap=noverlap, nfft=nfft, window=window, dc=dc, crop_freq=crop_freq, quantization_levels=quantization_bins, mode="wasserstein")
# s2g_detector_circularity = ut.S2GDetector(fs=fs, nperseg=nperseg, overlap=noverlap, nfft=nfft, window=window, dc=dc, crop_freq=crop_freq, quantization_levels=quantization_bins, mode="circularity")

fig = make_subplots(rows=4, 
                    cols=len(snr_values), 
                    shared_xaxes=True, 
                    vertical_spacing=0.05,
                    column_titles=[f"snr={snr} dB" for snr in snr_values])  #, 
                    # row_titles=["Spectrum", "Ref.", "Laplacian", "Wasserstein"])

for i, snr in enumerate(snr_values, start=1):
    rx = ut.add_noise_to_signal(raw_signal, snr_db=snr, fs=fs, signal_bw=1, noise_type='white')

    _F, _T, rx_Sxx, _phasogram = ut.calc_spectrogram(rx, fs, nperseg=nperseg, percent_overlap=noverlap, window=window, remove_dc=dc, crop_freq=crop_freq)
    # fig.add_trace(go.Heatmap(z=rx_Sxx.T, y=_T, x=_F, colorscale='Viridis', showlegend=False, showscale=False), row=1, col=i)

    rx_pxx, F_pxx = welch_detector.get_feature_vector(rx)
    fig.add_trace(go.Scatter(y=rx_pxx, x=F_pxx, showlegend=False, line=dict(color='blue')), row=1, col=i)

    rx_td_scores, F_td_scores = talmon_detector.get_feature_vector(rx)
    fig.add_trace(go.Scatter(y=rx_td_scores, x=F_td_scores, showlegend=False, line=dict(color='green')), row=2, col=i)

    Kl, F_Kl = s2g_detector_laplacian.get_feature_vector(rx)
    fig.add_trace(go.Scatter(y=Kl, x=F_Kl, showlegend=False, line=dict(color='purple')), row=3, col=i)

    Kw, F_Kw = s2g_detector_wasserstein.get_feature_vector(rx)
    fig.add_trace(go.Scatter(y=Kw, x=F_Kw, showlegend=False, line=dict(color='green')), row=4, col=i)

    # Kc, F_Kc = s2g_detector_circularity.get_feature_vector(rx)
    # fig.add_trace(go.Scatter(y=Kc, x=F_Kc, showlegend=False, line=dict(color='red')), row=5, col=i)

fig.update_xaxes(title_text='Frequency (Hz)', title_font=dict(size=20), row=4)
fig.update_yaxes(showticklabels=False)

fig.update_layout(
    height=600, width=1200,
    font=dict(size=20),
    title_font=dict(size=20),
    margin=dict(t=140, r=140),   # leave room for lifted column titles + shifted row titles
)

# Match column/row title offsets and bold them.
# Column titles sit above row 1 at y = 1 + offset; row titles sit to the
# right of the last column at x = 1 + offset (in paper coords).
title_offset = 0.06
column_title_texts = [f"snr={snr} dB" for snr in snr_values]
# row_title_texts = ["Spectrum", "Ref.", "Laplacian", "Wasserstein"]
for ann in fig.layout.annotations:
    ann.font = dict(size=20)
    if ann.text in column_title_texts:
        ann.text = f"<b>{ann.text}</b>"
        ann.y = 1 + title_offset
        ann.yanchor = 'bottom'
    # elif ann.text in row_title_texts:
    #     ann.text = f"<b>{ann.text}</b>"
    #     ann.x = 1 + title_offset
    #     ann.xanchor = 'left'
# fig.write_image("/Users/itamarmerfeld/Projects/UATR/results/simulation_example.svg")  #, scale=10)
fig.show()

---

# Quantization Size - Sanity Check

In [5]:
# simulation parameters
fs = 128000  # sampling frequency
f0 = 640  # signal frequency
duration = 60  # seconds
snr_values = [-2, -4, -6, -8]

# detector args
nperseg = 16000
nfft = None # 2*nperseg
noverlap = 0.5
window = 'hanning'
dc = 20
crop_freq = 8000
norm_size = 9
mode = "circularity"
quantization_bins = [5, 10, 100]

raw_signal = ut.simulate_raw_signal(f0, fs, duration)
F, T, raw_Sxx, raw_phasogram = ut.calc_spectrogram(raw_signal, fs, nfft=nfft, nperseg=nperseg, percent_overlap=noverlap/nperseg, window=window, remove_dc=dc, crop_freq=crop_freq)

# d2 = ut.S2GDetector(fs=fs, nperseg=nperseg, overlap=noverlap, nfft=nfft, window=window, dc=dc, crop_freq=crop_freq, quantization_levels=quantization_bins[0], mode=mode)
# d3 = ut.S2GDetector(fs=fs, nperseg=nperseg, overlap=noverlap, nfft=nfft, window=window, dc=dc, crop_freq=crop_freq, quantization_levels=quantization_bins[1], mode=mode)
d5 = ut.S2GDetector(fs=fs, nperseg=nperseg, overlap=noverlap, nfft=nfft, window=window, dc=dc, crop_freq=crop_freq, quantization_levels=quantization_bins[0], mode=mode)
d10 = ut.S2GDetector(fs=fs, nperseg=nperseg, overlap=noverlap, nfft=nfft, window=window, dc=dc, crop_freq=crop_freq, quantization_levels=quantization_bins[1], mode=mode)
d20 = ut.S2GDetector(fs=fs, nperseg=nperseg, overlap=noverlap, nfft=nfft, window=window, dc=dc, crop_freq=crop_freq, quantization_levels=quantization_bins[2], mode=mode)

detectors = [d5, d10, d20]

fig = make_subplots(rows=len(detectors), cols=len(snr_values), shared_xaxes=True, column_titles=[f"snr={snr} dB" for snr in snr_values], row_titles=[f"Q={q}" for q in quantization_bins])
for i, snr in enumerate(snr_values, start=1):
    rx = ut.add_noise_to_signal(raw_signal, snr_db=snr, fs=fs, signal_bw=1, noise_type='white')
    
    for j, d in enumerate(detectors, start=1):
        Kl, F_Kl = d.get_feature_vector(rx)
        fig.add_trace(go.Scatter(y=Kl, x=F_Kl, showlegend=False, line=dict(color='blue')), row=j, col=i)

fig.update_layout(height=800, width=300*len(snr_values), title_text=f"Detector mode: {mode} | Quantization levels: {quantization_bins}")
fig.show(renderer="browser")

---

## Main Simulation

### Welch

In [ ]:
# simulation parameters
fs = 128000  # sampling frequency
f0 = 640  # signal frequency
duration = 60  # seconds
snr_values = [-2, -4, -6, -8]
num_iterations = 100
tolerance = 5  # Hz
num_thresholds = 20

# detector args
nperseg = 16000
nfft = None # 2*nperseg
noverlap = 0.5
window = 'hanning'
dc = 20
crop_freq = 8000
norm_size = 9
mode = "welch"
quantization_bins = [2, 3, 5, 10, 20]
thresholds = np.linspace(0.01, 0.1, num_thresholds)

# results
all_pd_rates = np.zeros(len(snr_values), len(thresholds))
all_fa_rates = np.zeros(len(snr_values), len(thresholds))

# generate raw signal
raw_signal = ut.simulate_raw_signal(f0, fs, duration)

detector = ut.WelchDetector(fs=fs, nperseg=nperseg, overlap=noverlap, nfft=nfft, window=window, dc=dc, crop_freq=crop_freq, norm_size=norm_size)

for ti, threshold in enumerate(thresholds):

    for si, snr in enumerate(snr_values):
        pd = np.zeros(num_iterations)
        fa = np.zeros(num_iterations)
        for i in range(num_iterations):
            rx = ut.add_noise_to_signal(raw_signal, snr_db=snr, fs=fs, signal_bw=1, noise_type='white')

            is_detected, detections, (K, F) = detector.detect(rx, threshold)

            if is_detected:
                pd[i] = 1 if np.any((detections - f0) < tolerance) else 0
                fa[i] = 1 if np.any((detections - f0) > tolerance) else 0
            
        all_pd_rates[si, ti] = np.mean(pd)
        all_fa_rates[si, ti] = np.mean(fa)

        print(f"threshold={threshold}, snr={snr}, pd_rate={np.mean(pd)}, fa_rate={np.mean(fa)}")

print(f"completed simulation for all parameters with mode={mode}")

results = {"pd_rates": all_pd_rates, "fa_rates": all_fa_rates}
with open(f"../results/nb82_results_{mode}.pkl", "wb") as f:
    pickle.dump(results, f)

print(f"results saved to nb82_results_{mode}.pkl")


### S2G - Laplcian

In [ ]:
# simulation parameters
fs = 128000  # sampling frequency
f0 = 640  # signal frequency
duration = 60  # seconds
snr_values = [-2, -4, -6, -8]
num_iterations = 100
tolerance = 5  # Hz

# detector args
nperseg = 16000
nfft = None # 2*nperseg
noverlap = 0.5
window = 'hanning'
dc = 20
crop_freq = 8000
norm_size = 9
mode = "wasserstein"
quantization_bins = [2, 3, 5, 10, 20]

num_thresholds = 10
threshold_range = (0.2, 0.35)
thresholds = np.linspace(threshold_range[0], threshold_range[1], num_thresholds)

# results
all_pd_rates = np.zeros((len(quantization_bins), len(snr_values), len(thresholds)))
all_fa_rates = np.zeros((len(quantization_bins), len(snr_values), len(thresholds)))

# generate raw signal
raw_signal = ut.simulate_raw_signal(f0, fs, duration)

for qi, q in enumerate(quantization_bins):
    detector = ut.S2GDetector(fs=fs, nperseg=nperseg, overlap=noverlap, nfft=nfft, window=window, dc=dc, crop_freq=crop_freq, quantization_levels=q, mode=mode)

    for ti, threshold in enumerate(thresholds):

        for si, snr in enumerate(snr_values):
            pd = np.zeros(num_iterations)
            fa = np.zeros(num_iterations)
            for i in range(num_iterations):
                rx = ut.add_noise_to_signal(raw_signal, snr_db=snr, fs=fs, signal_bw=1, noise_type='white')

                is_detected, detections, (K, F) = detector.detect(rx, threshold)

                if is_detected:
                    pd[i] = 1 if np.any((detections - f0) < tolerance) else 0
                    fa[i] = 1 if np.any((detections - f0) > tolerance) else 0
                
            all_pd_rates[qi, si, ti] = np.mean(pd)
            all_fa_rates[qi, si, ti] = np.mean(fa)

            print(f"Q={q}, threshold={threshold}, snr={snr}, pd_rate={np.mean(pd)}, fa_rate={np.mean(fa)}")

print(f"completed simulation for all parameters with mode={mode}")

results = {"pd_rates": all_pd_rates, "fa_rates": all_fa_rates}
with open(f"../results/nb82_results_{mode}.pkl", "wb") as f:
    pickle.dump(results, f)

print(f"results saved to nb82_results_{mode}_with_Q_{quantization_bins}_and_{num_thresholds}_thresholds_in_range_{threshold_range}.pkl")

Q=2, threshold=0.2, snr=-2, pd_rate=0.0, fa_rate=0.0
Q=2, threshold=0.2, snr=-4, pd_rate=0.0, fa_rate=0.0
Q=2, threshold=0.2, snr=-6, pd_rate=0.0, fa_rate=0.0
Q=2, threshold=0.2, snr=-8, pd_rate=0.0, fa_rate=0.0
Q=2, threshold=0.21666666666666667, snr=-2, pd_rate=0.0, fa_rate=0.0
Q=2, threshold=0.21666666666666667, snr=-4, pd_rate=0.0, fa_rate=0.0
Q=2, threshold=0.21666666666666667, snr=-6, pd_rate=0.0, fa_rate=0.0
Q=2, threshold=0.21666666666666667, snr=-8, pd_rate=0.0, fa_rate=0.0
Q=2, threshold=0.23333333333333334, snr=-2, pd_rate=0.0, fa_rate=0.0
Q=2, threshold=0.23333333333333334, snr=-4, pd_rate=0.0, fa_rate=0.0
Q=2, threshold=0.23333333333333334, snr=-6, pd_rate=0.0, fa_rate=0.0
Q=2, threshold=0.23333333333333334, snr=-8, pd_rate=0.0, fa_rate=0.0
Q=2, threshold=0.25, snr=-2, pd_rate=0.0, fa_rate=0.0
Q=2, threshold=0.25, snr=-4, pd_rate=0.0, fa_rate=0.0
Q=2, threshold=0.25, snr=-6, pd_rate=0.0, fa_rate=0.0
Q=2, threshold=0.25, snr=-8, pd_rate=0.0, fa_rate=0.0
Q=2, threshold=0.266

### S2G - Wesserstain

In [ ]:
# simulation parameters
fs = 128000  # sampling frequency
f0 = 640  # signal frequency
duration = 60  # seconds
snr_values = [-2, -4, -6, -8]
num_iterations = 100
tolerance = 5  # Hz

# detector args
nperseg = 16000
nfft = None # 2*nperseg
noverlap = 0.5
window = 'hanning'
dc = 20
crop_freq = 8000
norm_size = 9
mode = "wasserstein"
quantization_bins = [2, 3, 5, 10, 20]

num_thresholds = 10
threshold_range = (0.02, 0.1)
thresholds = np.linspace(threshold_range[0], threshold_range[1], num_thresholds)

# results
all_pd_rates = np.zeros((len(quantization_bins), len(snr_values), len(thresholds)))
all_fa_rates = np.zeros((len(quantization_bins), len(snr_values), len(thresholds)))

# generate raw signal
raw_signal = ut.simulate_raw_signal(f0, fs, duration)

for qi, q in enumerate(quantization_bins):
    detector = ut.S2GDetector(fs=fs, nperseg=nperseg, overlap=noverlap, nfft=nfft, window=window, dc=dc, crop_freq=crop_freq, quantization_levels=q, mode=mode)

    for ti, threshold in enumerate(thresholds):

        for si, snr in enumerate(snr_values):
            pd = np.zeros(num_iterations)
            fa = np.zeros(num_iterations)
            for i in range(num_iterations):
                rx = ut.add_noise_to_signal(raw_signal, snr_db=snr, fs=fs, signal_bw=1, noise_type='white')

                is_detected, detections, (K, F) = detector.detect(rx, threshold)

                if is_detected:
                    pd[i] = 1 if np.any((detections - f0) < tolerance) else 0
                    fa[i] = 1 if np.any((detections - f0) > tolerance) else 0
                
            all_pd_rates[qi, si, ti] = np.mean(pd)
            all_fa_rates[qi, si, ti] = np.mean(fa)

            print(f"Q={q}, threshold={threshold}, snr={snr}, pd_rate={np.mean(pd)}, fa_rate={np.mean(fa)}")

print(f"completed simulation for all parameters with mode={mode}")

results = {"pd_rates": all_pd_rates, "fa_rates": all_fa_rates}
with open(f"../results/nb82_results_{mode}.pkl", "wb") as f:
    pickle.dump(results, f)

print(f"results saved to nb82_results_{mode}_with_Q_{quantization_bins}_and_{num_thresholds}_thresholds_in_range_{threshold_range}.pkl")

---

## Analysis

In [4]:
with open(f"../results/nb82_results_{mode}.pkl", "rb") as f:
    results = pickle.load(f)

pd_rates = results["pd_rates"]
fa_rates = results["fa_rates"]

fig = make_subplots(rows=len(snr_values), cols=1, row_titles=[f"SNR={snr}" for snr in snr_values])
for i, snr in enumerate(snr_values):
    for j, q in enumerate(quantization_bins):
        fig.add_trace(go.Scatter(x=fa_rates[j, i, :], y=pd_rates[j, i, :], mode="lines", name=f"Q={q}"), row=i+1, col=1)

fig.update_layout(height=800, width=300*len(snr_values), title_text=f"ROC Curve for {mode} Detector")
fig.show(renderer="browser")

python(18655) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
